In [1]:
import torch, transformers
import json
from copy import deepcopy

model_8B_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tok = transformers.AutoTokenizer.from_pretrained(model_8B_id)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_8B_id, dtype=torch.float16, device_map="cuda"
)

d:\GeoTKG\llama_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:26<00:00,  6.63s/it]


In [2]:
def get_test_data(path):
    examples = []
    # "D:\\GeoTKG\\cleandata\\tie\\test.json"
    with open(path, "r") as f:
        examples=[json.loads(line) for line in f]
    return examples

def get_ner_prompt(text, method):
    if method == "event and time":
        tags = "EVENT, DATE, TIME, DURATION, SET"
    elif method == "geoscience":
        tags = "LOCATION, MINERAL, ORE_DEPOSIT, ROCK, STRAT, TIMESCALE"
    sample = " ".join([wrd for sent in text for wrd in sent])
    sys = f'''You are a Named Entity Recognition (NER) system for tagging {method} entities. Identify and classify entities in the text based on the entity types: {tags}. 
            Each entity should be represented as a tuple (entity surface text, type) in valid JSON format.
            Return ONLY valid JSON (no markdown, no commentary). Escape double quotes as \".'''
    
    user = f'''Extract entities from the following text: {sample}'''
    messages = [
        {"role": "system", "content": sys},
        {"role": "user", "content": user}
    ]
    return messages

def get_norm_prompt(text, dct):
    sys = '''
        You are a time normalization system. Normalize the time expressions that have been tagged in the text using the document creation time as anchor for calendar time.
        Each time mention should be represented as a tuple: (surface text, normalized value).
        Calendar time expressions should be in ISO 8601 format (YYYY-MM-DD).
        Return ONLY valid JSON (no markdown, no commentary). Escape double quotes as \".
    '''
    user = f'Normalize time expressions from the following passage which has a document creation time of {dct}: {text}'
    messages = [
    {"role":"system","content":sys},
    {"role":"user","content":user}
    ]
    return messages

def get_et_ee_prompt(text):
    sys = '''
        You are a temporal linking system. Given a passage with annotated events (E#) and times (T#), link them and infer temporal relations.
        Event-Time relations should be outputed as ["E#", "T#"], if no time can be linked, infer its time indirectly using event-event relations.
        Event-Event temporal relations should be outputed as ["E#", "BEFORE|AFTER|OVERLAPS|CONTAINS|EQUALS|INDENTITY", "E#"] with each event pair appears at most once.
        Return ONLY valid JSON (no markdown, no commentary). 
    '''
    user = f'Link the events to times or events to events from the following passage: {text}'
    messages = [
    {"role":"system","content":sys},
    {"role":"user","content":user}
    ]
    return messages

def get_tkg_prompt(text, dct):
    text = " ".join([wrd for sent in text for wrd in sent])
    sys = '''
        You are an information extraction system for geoscience and general texts.
        Extract (1) times, (2) quintuples (events), and (3) temporal-relation triples.
        Return ONLY valid JSON (no markdown, no commentary). Escape double quotes as \".

        Gregorian Calendar Times and Geological Timescales
        - Each time has format: ["T#", "surface text", "normalized value or null", "DATE|DURATION|SET|TIME|GEO_TIME"]
        - Reuse the same T# if the surface text repeats.

        Quintuples (events)
        - Each event is one quintuple: ["E#", "subject or null", "event string", "object or null", "T# or null", "T# or null"]
        - Event string must be short (just the trigger words).
        - E# assigned in order of first mention; reuse IDs for duplicates.

        Temporal triples
        - BEFORE: event1 ends before event2 starts
        - AFTER:  event1 starts after event2 ends
        - DURING: event1 occurs fully within event2
        - CONTAINS: event1 fully contains event2
        - IDENTITY/EQUALS: same event/time span
        - OVERLAPS: partial intersection
        - Each relation is ["E#", "BEFORE|AFTER|DURING|CONTAINS|IDENTITY|EQUALS|OVERLAPS", "E#"]
        - Only E# allowed, never T#.
        - Each unordered event pair appears at most once.

        Validation
        - IDs sequential by first mention (E1, E2 ...; T1, T2 ...).
        - All T# in quintuples must exist in times.
        - JSON must be valid: no trailing commas, no comments.
    '''
    user = f'Extract events and temporal relations from the following passage (document creation time: {dct}):    {text}'
    messages = [
    {"role":"system","content":sys},
    {"role":"user","content":user}
    ]
    return messages

def jsonify(output):
    try:
        return json.loads(output)
    except json.JSONDecodeError:
        try:
            return json.loads(output+"}")
        except json.JSONDecodeError:
            return None

def inference(model, tok, prompt, max_new_tokens=4000):
    input_prompt = tok.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)
    inputs = tok(input_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=False)
    gen_tokens = out[0, inputs.input_ids.shape[-1]:]
    decodings = tok.decode(gen_tokens, skip_special_tokens=True)
    prediction = decodings.strip(":\n`").lstrip("```json\n").rstrip("\n```")
    return prediction

def norm_preprocess(text, times):
    out = ""
    for sn, sent in enumerate(text):
        sent_times = [t['offset'] for t in times if t['sent_id'] == sn]
        sent_times = sorted(sent_times, key=lambda x: x[0], reverse=True)

        for st, en in sent_times:
            sent.insert(en,"</timex>")
            sent.insert(st,"<timex>")
        out += " ".join(sent) + " "
    return out.strip()

def post_processing(preds):
    out = []
    for fn, pred in enumerate(preds):
        json_out = jsonify(pred['pred'])
        if json_out is None:
            testy = pred['pred'].partition('{')[-1].rpartition('}')[0]
            json_out = jsonify('{'+testy+'}')
        
        if json_out is None or json_out == {}:
            testy = pred['pred'].partition('{')[-1][:-6]
            json_out = jsonify('{'+testy+'}')

        if json_out is None:
            testy = pred['pred'].partition('{')[-1][:-1]
            json_out = jsonify('{'+testy+'}')

        if json_out is None:
            testy = pred['pred'].partition('{')[-1].rpartition(']')[0]
            json_out = jsonify('{'+testy+'}')

        if type(json_out) is list:
            json_out = json_out[0]

        keys = list(json_out.keys())
        keys.remove('times')
        keys.remove('quintuples')
        json_out['triples'] = json_out.pop(keys[0])

        out.append({'text':pred['text'], 'pred':json_out})
    return out

def ner_post_processing(ner_preds):
    poster = []

    for fn, pred in enumerate(ner_preds):
        if fn==121:
            continue
        stripped = pred['pred'].replace('{"entity surface text": ', '[').replace('"type": ', '').replace("}",']').replace('{"entity": ',"[").replace("{","[")
        texty = stripped.partition('[')[-1].rpartition(']')[0]
        pred_json = jsonify("["+texty+"]")
        if pred_json is None:
            print(fn)
            pred_json = texty
        poster.append({'text':pred['text'], 'pred':pred_json})
    return poster

def mark_e_t_in_text(sample):
    instances = [instance for instance in sample['instances'] if not(instance['type']!="EVENT" and instance['id']==0)]
    out = ""
    for sn, sent in enumerate(sample['text']):
        sent_cp = deepcopy(sent)
        sent_instances = [(i['offset'], i['id'], i['type']) for i in instances if i['sent_id'] == sn]
        sent_instances = sorted(sent_instances, key=lambda x: x[0], reverse=True)

        for (st, en), id, ty in sent_instances:
            if ty == "EVENT":
                sent_cp.insert(en,f"[/E{id}]")
                sent_cp.insert(st,f"[E{id}]")
            else:
                sent_cp.insert(en,f"[/T{id}]")
                sent_cp.insert(st,f"[T{id}]")
        out += " ".join(sent_cp) + " "
    return out.strip()

In [3]:
test_data = get_test_data("D:\\GeoTKG\\cleandata\\tie\\test.json")

In [4]:
def run_llama(prediction_type):
    chat_preds = []
    if prediction_type == "tkg":
        file_num = 1
        for example in test_data:
            dct = [inst['value'] for inst in example['instances'] if inst['type'] != "EVENT" and inst['id'] == 0][0]
            if len(example['text'])>50:
                half = int(len(example['text'])/2)
                prompt1 = get_tkg_prompt(example['text'][:half], dct)
                prompt2 = get_tkg_prompt(example['text'][half:], dct)
                prediction = [inference(model, tok, prompt1), inference(model, tok, prompt2)]
            else:
                prompt = get_tkg_prompt(example['text'], dct)
                prediction = inference(model, tok, prompt)
            chat_preds.append({"text":example['text'], "pred":prediction})
            print(f"Processed example {file_num} / {len(test_data)}")
            file_num += 1
    elif prediction_type == "ner":
        print("----- RUNNING NER PREDICTIONS -----")
        file_num = 1
        for example in test_data:
            if len(example['text'])>50:
                half = int(len(example['text'])/2)
                prompt1 = get_ner_prompt(example['text'][:half], "event and time")
                prompt2 = get_ner_prompt(example['text'][half:], "event and time")
                prediction = [inference(model, tok, prompt1, max_new_tokens=1000), inference(model, tok, prompt2, max_new_tokens=1000)]
            else:
                prompt = get_ner_prompt(example['text'], "event and time")
                prediction = inference(model, tok, prompt, max_new_tokens=1000)
            chat_preds.append({"text":example['text'], "pred":prediction})
            print(f"Processed example {file_num} / {len(test_data)}")
            file_num += 1
    elif prediction_type == "norm":
        file_num = 1
        for example in test_data:
            dct = [inst['value'] for inst in example['instances'] if inst['type'] != "EVENT" and inst['id'] == 0][0]
            prompt = get_norm_prompt(norm_preprocess(example['text'], [instance for instance in example['instances'] if instance['type'] != "EVENT" and instance['id'] != 0]), dct)
            prediction = inference(model, tok, prompt, max_new_tokens=500)
            chat_preds.append({"text":example['text'], "pred":prediction})
            print(f"Processed example {file_num} / {len(test_data)}")
            file_num += 1
    return chat_preds

In [5]:
chat_preds = run_llama("norm")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed example 1 / 602


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed example 2 / 602


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Processed example 599 / 602


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed example 600 / 602


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processed example 601 / 602
Processed example 602 / 602


In [6]:
with open("llama3-8B-norm-preds.json", 'w') as json_file:
    for sample in chat_preds:
        json_file.write(json.dumps(sample)+"\n")